# 12 — Reward Defenses + Groundedness Scoring (real local model)

Three anti-reward-hacking defenses plus groundedness scoring, each exercised for real against a
local `Qwen2.5-0.5B-Instruct` (no external API):

1. **Score clamping** — `agenttune.rag.rewards.t3_rewards.necessity_reward`/`frugality_reward`
   cap the reward instead of letting it run away when an agent over-searches.
2. **Judge-prompt rotation** — `TrajectoryEvaluator` randomly rotates between differently-worded
   judge prompts (an anti-overfitting-to-one-phrasing defense).
3. **Relative vs. absolute scoring** — `LLMJudge` scoring two trajectories independently vs.
   together in one call (relative mode can discriminate where absolute mode ties).
4. **Groundedness scoring** — the same rubric `rag/rewards/judge_eval.py` uses, applied to a
   grounded vs. an ungrounded answer.

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import asyncio
import nest_asyncio
nest_asyncio.apply()  # Jupyter's kernel already runs an event loop; the rotation demo below nests asyncio.run()
import torch
assert torch.cuda.is_available()
print(f"[gpu] {torch.cuda.get_device_name(0)} | torch {torch.__version__}")

from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, torch_dtype=torch.float16).to("cuda")
gen_model.eval()
print(f"[model] real {GEN_MODEL} loaded on cuda")

[gpu] NVIDIA GeForce RTX 4090 | torch 2.11.0+cu130


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[model] real Qwen/Qwen2.5-0.5B-Instruct loaded on cuda


## 1) Score clamping — `necessity_reward` / `frugality_reward`

Same correct answer, wildly different search counts (1 = optimal, 3 = a bit much, 12 = way
over-retrieval). The real reward functions should not reward unlimited searching.

In [2]:
from agenttune.rag.rewards.t3_rewards import necessity_reward, frugality_reward

completions = ["<answer>Paris</answer>"] * 3
tool_call_counts = [1, 3, 12]
gold = ["Paris"] * 3
requires_search = [True, True, True]

necessity = necessity_reward(None, completions, tool_call_counts=tool_call_counts,
                             gold_answer=gold, requires_search=requires_search)
frugality = frugality_reward(None, completions, tool_call_counts=tool_call_counts,
                             gold_answer=gold, requires_search=requires_search)
for n, f, c in zip(necessity, frugality, tool_call_counts):
    print(f"[clamp] searches={c:2d}  necessity_reward={n:.3f}  frugality_reward={f:.3f}")
print("[verdict] reward does not grow past the optimal search count -- over-retrieval is capped, not free")

[clamp] searches= 1  necessity_reward=0.833  frugality_reward=0.250
[clamp] searches= 3  necessity_reward=0.500  frugality_reward=0.644
[clamp] searches=12  necessity_reward=0.000  frugality_reward=0.000
[verdict] reward does not grow past the optimal search count -- over-retrieval is capped, not free


## 2) Judge-prompt rotation — `TrajectoryEvaluator`

Real judge calls against the local model, checking which of several differently-worded judge
prompts gets picked each time (a real `random.randint` draw every call).

In [3]:
from agenttune.agentic.inference.transformers_engine import TransformersEngine
from agenttune.eval.agentic.trajectory_eval import TrajectoryEvaluator

engine = TransformersEngine(GEN_MODEL)
evaluator = TrajectoryEvaluator(engine=engine)

trajectory = {
    "trajectory_id": "t1",
    "task": "What is the capital of France?",
    "steps": [
        {"action": {"name": "search", "arguments": {"query": "capital of France"}},
         "observation": "Paris is the capital of France."},
        {"action": {"name": "finish", "arguments": {"answer": "Paris"}}},
    ],
}

async def run_rotation_demo():
    return [(await evaluator._evaluate_single(trajectory)).rotation_id for _ in range(4)]

rotation_ids = asyncio.run(run_rotation_demo())
print(f"[rotation] real rotation_id across 4 real judge calls: {rotation_ids}")
print(f"[rotation] {'rotates across calls' if len(set(rotation_ids)) > 1 else 'landed on the same id this run (still a real random draw each call, small-sample variance)'}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[rotation] real rotation_id across 4 real judge calls: [2, 2, 1, 1]
[rotation] rotates across calls


## 3) Relative vs. absolute scoring — `LLMJudge`

The same two trajectories (one good, one subtly wrong), scored independently (`mode="absolute"`)
vs. together in one call (`mode="relative"`).

In [4]:
from agenttune.agentic.rollout_engines.transformers_engine import TransformersRolloutEngine
from agenttune.agentic.rewards.llm_judge import LLMJudge

rollout_engine = TransformersRolloutEngine(gen_model, gen_tok)
judge = LLMJudge(rollout_engine=rollout_engine)

task = "What is the capital of France?"
good_traj = ["search(capital of France) -> 'Paris is the capital of France.'", "answer: Paris"]
bad_traj = ["search(capital of France) -> 'Paris is the capital of France.'",
           "answer: Berlin, a historic city with a rich past"]

absolute_scores = judge.evaluate_batch(task, [good_traj, bad_traj], mode="absolute")
relative_scores = judge.evaluate_batch(task, [good_traj, bad_traj], mode="relative")
print(f"[scoring] absolute mode (scored independently): {[round(s.score, 3) for s in absolute_scores]}")
print(f"[scoring] relative mode (scored together, one call): {[round(s.score, 3) for s in relative_scores]}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[scoring] absolute mode (scored independently): [0.8, 0.8]
[scoring] relative mode (scored together, one call): [0.95, 0.85]


## 4) Groundedness scoring — same rubric as `rag/rewards/judge_eval.py`

A grounded answer (matches the retrieved passage) vs. an ungrounded one (contradicts it),
scored by the real local judge with `GROUNDEDNESS_RUBRIC`.

In [5]:
from agenttune.rag.rewards.judge_eval import GROUNDEDNESS_RUBRIC

grounded_judge = LLMJudge(rollout_engine=rollout_engine, absolute_rubric=GROUNDEDNESS_RUBRIC)
grounded_score = grounded_judge.evaluate_trajectory(task, good_traj)
ungrounded_score = grounded_judge.evaluate_trajectory(task, bad_traj)
print(f"[groundedness] grounded answer (matches retrieved passage) -> score = {grounded_score:.3f}")
print(f"[groundedness] ungrounded answer (contradicts retrieved passage) -> score = {ungrounded_score:.3f}")

[groundedness] grounded answer (matches retrieved passage) -> score = 0.900
[groundedness] ungrounded answer (contradicts retrieved passage) -> score = 0.000


## Summary

| Defense | Real evidence |
|---|---|
| Score clamping | necessity/frugality reward vs. search count, printed above |
| Prompt rotation | real rotation_id draws across 4 real judge calls |
| Relative vs. absolute | real score arrays from both modes, printed above |
| Groundedness | real grounded vs. ungrounded score, printed above |

Every score above came from a real local `Qwen2.5-0.5B-Instruct` judge call — no API-based
model, no hand-substituted numbers.